In [9]:
import pandas as pd
import ast

def parse_genres(genres):
    if isinstance(genres, str):
        try:
            genres_list = [g['name'] for g in ast.literal_eval(genres)]
            return " ".join(genres_list)
        except:
            return genres.replace('|', ' ')
    return ""

# Load and preprocess
movies = pd.read_csv("movies.csv")
movies.dropna(inplace=True)

movies['tags'] = movies.apply(
    lambda row: f"{row['title']} {parse_genres(row['genres'])}".lower(),
    axis=1
)
movies = movies[movies['tags'].notnull()]
movies = movies[movies['tags'].str.strip() != ""]

new_df = movies[['movieId', 'title', 'tags']].copy()
new_df['tag_set'] = new_df['tags'].apply(lambda x: set(x.split()))

def jaccard_similarity(set1, set2):
    intersection = len(set1 & set2)
    union = len(set1 | set2)
    if union == 0:
        return 0
    return intersection / union

def similarity_to_stars(sim_score, max_stars=5):
    # Convert similarity (0.0 to 1.0) to integer stars (0 to max_stars)
    return f"{int(round(sim_score * max_stars))}/{max_stars}"

def recommend(movie):
    # Case-insensitive search
    movie_lower = movie.lower()
    match = new_df[new_df['title'].str.lower() == movie_lower]
    if match.empty:
        print("❌ Movie not found! Check the title.")
        return
    movie_index = match.index[0]
    target_set = new_df.loc[movie_index, 'tag_set']
    similarities = []
    for idx, row in new_df.iterrows():
        if idx == movie_index:
            continue
        sim = jaccard_similarity(target_set, row['tag_set'])
        similarities.append((idx, sim))
    top5 = sorted(similarities, reverse=True, key=lambda x: x[1])[:5]
    print(f"\n🎬 Top 5 Recommended Movies similar to '{movie}':")
    for idx, sim in top5:
        print(f"👉 {new_df.loc[idx, 'title']} (Score: {similarity_to_stars(sim)})")

def recommend_by_genre(genre):
    genre = genre.lower()
    matches = new_df[new_df['tags'].str.contains(genre)]
    if matches.empty:
        print(f"❌ No movies found for genre '{genre}'.")
    else:
        print(f"\n🎬 Top 20 Movies in the genre '{genre}':")
        for title in matches['title'].head(20):
            print("👉", title)

print("\n✅ Ready for recommendations!")
recommend("Sabrina (1995)")  # Change as needed 
recommend("Four Rooms (1995)")
recommend_by_genre("romance") 


✅ Ready for recommendations!

🎬 Top 5 Recommended Movies similar to 'Sabrina (1995)':
👉 Clueless (1995) (Score: 3/5)
👉 Mallrats (1995) (Score: 3/5)
👉 Sabrina (1954) (Score: 3/5)
👉 Nine Months (1995) (Score: 2/5)
👉 Forget Paris (1995) (Score: 2/5)

🎬 Top 5 Recommended Movies similar to 'Four Rooms (1995)':
👉 Friday (1995) (Score: 2/5)
👉 Rent-a-Kid (1995) (Score: 2/5)
👉 Coldblooded (1995) (Score: 2/5)
👉 Angus (1995) (Score: 2/5)
👉 Honeymoons (1995) (Score: 2/5)

🎬 Top 20 Movies in the genre 'romance':
👉 Grumpier Old Men (1995)
👉 Waiting to Exhale (1995)
👉 Sabrina (1995)
👉 American President, The (1995)
👉 Cutthroat Island (1995)
👉 Sense and Sensibility (1995)
👉 Leaving Las Vegas (1995)
👉 Persuasion (1995)
👉 Wings of Courage (1995)
👉 Carrington (1995)
👉 Clueless (1995)
👉 How to Make an American Quilt (1995)
👉 Pocahontas (1995)
👉 When Night Is Falling (1995)
👉 Mighty Aphrodite (1995)
👉 Postman, The (Postino, Il) (1994)
👉 Two if by Sea (1996)
👉 French Twist (Gazon maudit) (1995)
👉 Bed of Ro